## Libraries used

## Reproducibility note

This public portfolio copy removes stored execution outputs and uses the official validation split for training decisions. The test split is reserved for final evaluation. Metrics quoted in the repository README come from the original academic report and have not been regenerated from this sanitized copy.


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score
from torch.functional import F
import numpy as np
import medmnist
from medmnist import INFO, Evaluator
from torch.utils.data import WeightedRandomSampler
from sklearn.utils import resample
from torch.utils.data import random_split, DataLoader

# 2D dataset with size 28x28

In [ ]:
data_flag = 'dermamnist'
download = True
BATCH_SIZE = 128

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])


## Reading MedMNIST data, preprocessing and encapsulate it into dataloader form.

In [ ]:
# Preprocessing pipeline for input images
data_transform = transforms.Compose([
    transforms.ToTensor(),  # Convert image to PyTorch tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Normalize using ImageNet means
                         std=[0.229, 0.224, 0.225])   # Normalize using ImageNet stds
])

# Load the official training, validation, and test splits
train_dataset = DataClass(split='train', transform=data_transform, download=download)
val_dataset = DataClass(split='val', transform=data_transform, download=download)
test_dataset = DataClass(split='test', transform=data_transform, download=download)

# Class distribution (number of samples per class)
class_distribution = [228, 359, 769, 80, 779, 4693, 99]
total_samples = sum(class_distribution)

# Calculate class weights to handle class imbalance
class_weights = [total_samples / (len(class_distribution) * count) for count in class_distribution]
class_weights = torch.FloatTensor(class_weights)
print("Class Weights:", class_weights)

# Create a list of sample indices for each class
class_indices = [[] for _ in range(len(class_distribution))]
temp_loader = DataLoader(train_dataset, batch_size=128, shuffle=False)
start_idx = 0

# Build mapping from sample indices to class labels
for batch_data, batch_labels in temp_loader:
    for i, label in enumerate(batch_labels):
        class_idx = label.item()
        class_indices[class_idx].append(start_idx + i)
    start_idx += len(batch_labels)

# Assign sample weights based on class weights
sample_weights = torch.zeros(len(train_dataset))
for class_idx, indices in enumerate(class_indices):
    for idx in indices:
        sample_weights[idx] = class_weights[class_idx]

# Create a sampler for balanced sampling during training
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

#pil_dataset = DataClass(split='train', download=download)  # Version without transforms

# Use weighted sampling only for training; evaluation loaders remain deterministic
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

# fit_dataloader: A Training Loop for PyTorch Models

In [ ]:
# Training function using dataloader
def fit_dataloader(train_loader, model, loss_fn, optimizer, n_epochs, device, val_loader):
    model.to(device)
    train_losses, train_accuracies = [], []
    val_losses, val_accuracies = [], []
    
    # Early stopping and best model tracking
    best_model_wts = None
    best_val_loss = float('inf')
    best_epoch = 0
    epochs_no_improve = 0
    early_stopping_patience = 15

    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        all_preds, all_targets = [], []

        # Iterate over batches
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).squeeze(1).long()  # Ensure correct shape for loss function
            optimizer.zero_grad()
            outputs = model(X_batch)
            
            # Apply log-softmax for NLLLoss if needed
            if isinstance(loss_fn, nn.NLLLoss):
                outputs = F.log_softmax(outputs, dim=1)
            loss = loss_fn(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
            # Track predictions and labels
            _, preds = torch.max(outputs, 1)
            all_preds.append(preds.cpu())
            all_targets.append(y_batch.cpu())

        # Compute average loss and accuracy for training
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()
        train_acc = accuracy_score(all_targets, all_preds)
        train_accuracies.append(train_acc)

        # Validation phase
        if val_loader:
            model.eval()
            val_loss = 0.0
            val_preds, val_targets = [], []

            with torch.no_grad():
                for X_val, y_val in val_loader:
                    X_val = X_val.to(device)
                    y_val = y_val.to(device).squeeze(1).long()
                    outputs = model(X_val)
                    if isinstance(loss_fn, nn.NLLLoss):
                        outputs = F.log_softmax(outputs, dim=1)
                    loss = loss_fn(outputs, y_val)
                    val_loss += loss.item()

                    _, preds = torch.max(outputs, 1)
                    val_preds.append(preds.cpu())
                    val_targets.append(y_val.cpu())

            # Average validation loss and accuracy
            val_loss /= len(val_loader)
            val_losses.append(val_loss)
            val_preds = torch.cat(val_preds).numpy()
            val_targets = torch.cat(val_targets).numpy()
            val_acc = accuracy_score(val_targets, val_preds)
            val_accuracies.append(val_acc)

            print(f"Epoch [{epoch+1}/{n_epochs}], Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

            # Save best model based on validation loss
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                best_model_wts = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= early_stopping_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
        else:
            print(f"Epoch [{epoch+1}/{n_epochs}], Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}")

    # Load best model weights
    if best_model_wts:
        model.load_state_dict(best_model_wts)

    # Return metrics and best model
    metrics = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,
        "best_epoch": best_epoch
    }

    return metrics, model.cpu()

# evaluate_network: Performance Metrics for Model Evaluation (Evaluates network's performance)

In [ ]:
# Evaluation function for model performance
def evaluate_network(model, dataloader, device):
    model.to(device)
    model.eval()
    all_preds, all_targets = [], []

    # Iterate through data and gather predictions
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).squeeze(1).long()
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.append(preds.cpu())
            all_targets.append(y_batch.cpu())

    # Concatenate and evaluate metrics
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    conf_mat = confusion_matrix(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='weighted')
    class_report = classification_report(all_targets, all_preds, digits=2)

    print("Confusion Matrix:\n", conf_mat)
    print("\nClassification Report:\n", class_report)
    print("F1 Score:", f1)

    return conf_mat, f1, class_report

# CNN class implementation 

In [ ]:
# Define a CNN model using PyTorch's nn.Module
class CNN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(CNN, self).__init__()

        # First convolutional block: Conv -> BatchNorm -> ReLU -> MaxPool
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # Second convolutional block
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # Third convolutional block
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)

        # Adaptive average pooling to reduce to fixed size
        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))

        # Flatten feature maps for the fully connected layers
        self.flatten = nn.Flatten()

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Apply conv layers and pooling
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)

        # Apply dropout and flatten for dense layers
        x = self.dropout(x)
        x = self.avgpool(x)
        x = self.flatten(x)

        # Fully connected layers with ReLU in between
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# CNN class training, plotting loss values 

In [ ]:
optimizers = {
    "Adam": lambda model: optim.Adam(model.parameters(), lr=0.001),
    "RMSprop": lambda model: optim.RMSprop(model.parameters(), lr=0.001, alpha=0.9),
    "SGD": lambda model: optim.SGD(model.parameters(), lr=0.001, momentum=0.9),
    "AdamW": lambda model: optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5),
}

loss_functions = {
    "CrossEntropyLoss": nn.CrossEntropyLoss(),
    "MultiMarginLoss": nn.MultiMarginLoss(),
    "NLLLoss": nn.NLLLoss(),
}

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


def plot_training_curves(metrics):
    train_losses = metrics["train_losses"]
    val_losses = metrics["val_losses"]
    train_accuracies = metrics["train_accuracies"]
    val_accuracies = metrics["val_accuracies"]
    best_epoch = metrics.get("best_epoch")
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label="Train Loss", marker="o")
    plt.plot(epochs, val_losses, label="Validation Loss", marker="o")
    if best_epoch:
        plt.axvline(x=best_epoch, color="r", linestyle="--", label=f"Best Epoch: {best_epoch}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label="Train Accuracy", marker="o")
    plt.plot(epochs, val_accuracies, label="Validation Accuracy", marker="o")
    if best_epoch:
        plt.axvline(x=best_epoch, color="r", linestyle="--", label=f"Best Epoch: {best_epoch}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curve")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def run_experiments():
    """Select a configuration on validation data and evaluate test data once."""
    best_validation_score = 0.0
    best_combination = None
    best_model = None
    all_metrics = {}

    for optimizer_name, optimizer_factory in optimizers.items():
        for loss_name, loss_function in loss_functions.items():
            print(f"\nTraining with {optimizer_name} and {loss_name}...")
            cnn = CNN(input_channels=3, num_classes=7)
            optimizer = optimizer_factory(cnn)

            metrics, trained_model = fit_dataloader(
                train_loader,
                cnn,
                loss_function,
                optimizer,
                n_epochs=80,
                device=device,
                val_loader=val_loader,
            )
            key = f"{optimizer_name}_{loss_name}"
            all_metrics[key] = metrics

            print("\nEvaluating configuration on the validation split...")
            _, validation_f1, _ = evaluate_network(trained_model, val_loader, device)
            if validation_f1 > best_validation_score:
                best_validation_score = validation_f1
                best_combination = (optimizer_name, loss_name)
                best_model = copy.deepcopy(trained_model)

    if best_model is None or best_combination is None:
        raise RuntimeError("No CNN configuration completed successfully")

    best_key = f"{best_combination[0]}_{best_combination[1]}"
    plot_training_curves(all_metrics[best_key])

    print("\nFinal evaluation on the held-out test split...")
    _, test_f1, _ = evaluate_network(best_model, test_loader, device)
    return best_combination, best_validation_score, test_f1, all_metrics


best_combination, best_validation_f1, final_test_f1, experiment_metrics = run_experiments()
print(f"\nBest validation configuration: {best_combination}")
print(f"Validation F1: {best_validation_f1:.4f}")
print(f"Held-out test F1: {final_test_f1:.4f}")
